# H&M Fashion Recommendations — Simple Start

This notebook loads the data, understands it, and shows how training looks on it.

Goal: understand what each file contains, how they connect, and how a model learns from them.

## 1. Setup — import libraries and set paths

In [ ]:
import pandas as pd
import os

DATA_DIR = r'A:\IDEs\PyCharm\hAnd-m-PersonalizedFashionRecommendations\dataset'

print('Files in dataset folder:')
for f in os.listdir(DATA_DIR):
    print(f'  {f}')

## 2. articles.csv — what products exist

Each row is one product. Key column: `article_id`.

In [ ]:
articles = pd.read_csv(os.path.join(DATA_DIR, 'articles.csv'))

print(f'Shape: {articles.shape[0]} rows, {articles.shape[1]} columns')
print(f'\nColumn names:')
for col in articles.columns:
    print(f'  {col}')

articles[['article_id', 'prod_name', 'product_type_name', 'colour_group_name', 'department_name']].head(3)

## 3. customers.csv — who the customers are

Each row is one customer. Key column: `customer_id`.

In [ ]:
customers = pd.read_csv(os.path.join(DATA_DIR, 'customers.csv'))

print(f'Shape: {customers.shape[0]} rows, {customers.shape[1]} columns')
print(f'\nColumn names:')
for col in customers.columns:
    print(f'  {col}')

customers.head(3)

## 4. transactions_train.csv — what people bought

The most important file. Each row is one purchase.

It connects customers to articles: customer bought a product on a date for a price.

Full file is 31.8M rows (3.2GB). We load a sample to keep it fast.

In [ ]:
transactions = pd.read_csv(os.path.join(DATA_DIR, 'transactions_train.csv'), nrows=100000)

print(f'Shape (sample): {transactions.shape[0]} rows, {transactions.shape[1]} columns')
print(f'\nColumn names:')
for col in transactions.columns:
    print(f'  {col}')

transactions.head(3)

## 5. sample_submission.csv — what we need to output

Each customer gets 12 article predictions (space-separated).

In [ ]:
submission = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

print(f'Shape: {submission.shape[0]} rows, {submission.shape[1]} columns')
print(f'\nColumn names:')
for col in submission.columns:
    print(f'  {col}')

submission.head(3)

## 6. How the 5 files connect

```
customers.csv          transactions_train.csv          articles.csv
─────────────          ─────────────────────           ───────────
customer_id  ──────►   customer_id                     article_id  ◄────── article_id
                      article_id    ──────────────►
                      t_dat, price, sales_channel_id
```

- transactions connects customers to articles (who bought what)
- customers.csv tells us about the person
- articles.csv tells us about the product
- images/ shows us what the product looks like
- sample_submission.csv is the form we fill with predictions

In [ ]:
# Merge: 5 purchases with customer + article info side by side
sample = transactions.head(5).merge(customers, on='customer_id').merge(articles, on='article_id')

print('Merged view (5 purchases with everything together):')
print()
sample[['t_dat', 'customer_id', 'article_id', 'age', 'prod_name', 'product_type_name', 'colour_group_name', 'price']].head(5)

---

# Part 2: How Training Works on This Data

Now we see how a model learns from this data. We go from simplest to slightly more complex.

The code below assumes data is clean (no missing values, no issues). In reality you'd clean first, but we skip that here.

## 7. Split the data by time

We can't split randomly like normal ML. Purchases happen over time. We train on old data, test on recent data.

- Train = everything before a cutoff date
- Test = the 7 days right after the cutoff (this simulates the real competition task)

Think of it like: study January to September, then test if you can predict October's first week.

In [ ]:
# Pretend we loaded the FULL transactions (not just 100K sample)
# In real code: transactions = pd.read_csv(...full file...)

# Convert date column to datetime
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])

# Find the last date in the data
last_date = transactions['t_dat'].max()
print(f'Last date in data: {last_date}')

# Split: train = everything before last 7 days, test = last 7 days
cutoff = last_date - pd.Timedelta(days=7)

train = transactions[transactions['t_dat'] < cutoff]
test  = transactions[transactions['t_dat'] >= cutoff]

print(f'Train: {len(train)} purchases (before {cutoff.date()})')
print(f'Test:  {len(test)} purchases (after {cutoff.date()})')
print(f'\nThis is our split:')
print(f'  [--------- TRAIN (old purchases) ---------] [TEST (7 days)]')
print(f'  start ................................. cutoff ........... last_date')

## 8. Simplest model: Popularity baseline

The simplest recommendation: find what products were bought the most, recommend those to everyone.

No machine learning. Just counting. But it's a real baseline that competitions use.

**How it works:**
1. Count how many times each article was purchased in the train data
2. Sort by count (most popular first)
3. Take the top 12
4. Recommend those same 12 to every customer

In [ ]:
# Step 1: Count how many times each article was bought
article_counts = train['article_id'].value_counts()

print('Top 12 most popular articles:')
top_12 = article_counts.head(12)
print(top_12)

# Step 2: These are our predictions for EVERY customer
popular_12 = list(top_12.index)
print(f'\nWe will recommend these 12 article IDs to every customer:')
print(popular_12)

## 9. Make predictions with the popularity model

Take the top 12 popular articles and assign them to every customer in the submission file.

In [ ]:
# The submission file has all 1.37M customer IDs
# We give each customer the same 12 popular articles

# Convert the 12 article IDs to a space-separated string (that's the submission format)
prediction_str = ' '.join(str(aid) for aid in popular_12)

print(f'Prediction string (same for all customers):')
print(prediction_str)

# Fill the submission
submission['prediction'] = prediction_str

print(f'\nSubmission preview:')
submission.head(5)

# Save to file
# submission.to_csv('submission_popularity.csv', index=False)
# print('Saved! This is a valid submission.')

## 10. Evaluate: did our predictions match?

We have the test data (the real 7-day purchases). We check: did any of our 12 recommended articles actually appear in what customers bought?

The metric is MAP@12 (Mean Average Precision @ 12). In simple terms:
- For each customer, check how many of your 12 predictions they actually bought
- Reward you more if the correct item is ranked higher (position 1 > position 12)
- Average across all customers

In [ ]:
# Build ground truth: what each customer ACTUALLY bought in the test period
test_purchases = test.groupby('customer_id')['article_id'].apply(list).to_dict()

print(f'Customers who bought something in test period: {len(test_purchases)}')
print(f'Example: customer bought these articles: {list(test_purchases.values())[0] if test_purchases else "none"}')

# Simple MAP@12 calculation
def map_at_12(predictions, ground_truth):
    """
    predictions: list of 12 article IDs we recommended
    ground_truth: list of article IDs the customer actually bought
    """
    if not ground_truth:
        return 0.0
    
    score = 0.0
    hits = 0
    for i, pred in enumerate(predictions):
        if pred in ground_truth:
            hits += 1
            score += hits / (i + 1)  # precision at this position
    
    return score / min(len(ground_truth), 12)

# Evaluate for all customers in test
scores = []
for customer_id, actual_bought in test_purchases.items():
    score = map_at_12(popular_12, actual_bought)
    scores.append(score)

mean_ap = sum(scores) / len(scores) if scores else 0
print(f'\nMAP@12 score: {mean_ap:.4f}')
print(f'This means: on average, {(mean_ap*100):.1f}% of our predictions were correct and well-ranked')

## 11. Next step up: Collaborative Filtering

Popularity gives everyone the same 12 items. That's boring.

Collaborative filtering says: **"customers who bought similar things to you also bought X"**

How it works:
1. Build a matrix: rows = customers, columns = articles, values = how many times they bought it
2. The model learns hidden patterns (customer A and B are similar because they both bought dresses)
3. For a new customer, find similar customers and recommend what they bought

This is the simplest ML approach for recommendations.

In [ ]:
# Step 1: Build a customer-article matrix
# Each row = a customer, each column = an article, value = number of purchases

# We use only train data (not test)
interaction_counts = train.groupby(['customer_id', 'article_id']).size().reset_index(name='count')

print(f'Customer-article interactions: {len(interaction_counts)}')
print(f'Example: customer bought article X this many times:')
print(interaction_counts.head(5))

# This matrix is HUGE and SPARSE (mostly zeros)
# 1.37M customers x 105K articles = 144 billion cells, but most are 0
# That's why we use special libraries like `implicit` that handle sparse data
print(f'\nMatrix size would be: {customers.shape[0]} x {articles.shape[0]} = {customers.shape[0] * articles.shape[0]:,} cells')
print(f'But only {len(interaction_counts)} cells have data (that\'s {len(interaction_counts)/(customers.shape[0]*articles.shape[0])*100:.4f}% non-zero)')

In [ ]:
# Step 2: Train ALS (Alternating Least Squares) — the most common CF algorithm

# ALS works like this:
# - It creates a vector (embedding) for each customer and each article
# - Similar customers get similar vectors, similar articles get similar vectors
# - To recommend: find articles whose vector is closest to the customer's vector

# This is what the training code looks like (using the `implicit` library):

from implicit.als import ALSRecommender
import scipy.sparse as sparse

# # Convert article_id and customer_id to numbers (the model needs integers, not strings)
# article_ids = train['article_id'].unique()
# article_to_idx = {aid: i for i, aid in enumerate(article_ids)}
# customer_ids = train['customer_id'].unique()
# customer_to_idx = {cid: i for i, cid in enumerate(customer_ids)}

# # Build sparse matrix (customer x article)
# rows = train['customer_id'].map(customer_to_idx).values
# cols = train['article_id'].map(article_to_idx).values
# vals = train.groupby(['customer_id', 'article_id'])['count'].sum().values  # purchase counts
# sparse_matrix = sparse.csr_matrix((vals, (rows, cols)), shape=(len(customer_ids), len(article_ids)))

# # Train the model
# model = ALSRecommender(factors=50, regularization=0.01, iterations=20)
# model.fit(sparse_matrix)

# # Now the model has learned a 50-dimension vector for each customer and each article
# # To recommend for a customer:
# #   recommendations = model.recommend(customer_idx, sparse_matrix[customer_idx], n=12)

print('This is commented out because it needs the full dataset.')
print('But this is the SHAPE of how collaborative filtering training looks.')
print()
print('The key idea:')
print('  1. Build a customer x article matrix (who bought what)')
print('  2. Train ALS to learn vectors for each customer and article')
print('  3. Recommend by finding closest article vectors to a customer vector')
print('  4. Take top 12 as your predictions')

## 12. More advanced: Neural Network (Two-Tower)

Collaborative filtering only uses purchase history. But we also have customer age, product color, product type, images, etc.

A neural network can use ALL of that.

**Two-Tower model:**
- Tower 1 takes customer features (age, club status, etc.) → outputs a vector
- Tower 2 takes article features (type, color, department, etc.) → outputs a vector
- Dot product of the two vectors = how likely this customer buys this article
- Train on real purchases (positive) + random articles they didn't buy (negative)

This is what the plan in your Obsidian vault describes. But it's advanced — start with popularity, then CF, then this.

In [ ]:
# What Two-Tower training looks like (pseudocode — not runnable):

print("""
Two-Tower training shape:
─────────────────────────

# 1. Prepare features
customer_features = [age, club_status, fashion_news, active]
article_features  = [product_type, color, department, garment_group]

# 2. Build the model (TensorFlow/Keras)
customer_tower = keras.Sequential([
    keras.layers.Input(4),           # 4 customer features
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(64),                        # 64-dim customer vector
])

article_tower = keras.Sequential([
    keras.layers.Input(4),           # 4 article features
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(64),                        # 64-dim article vector
])

# 3. Score = dot product of customer vector and article vector
#    High score = likely to buy, low score = unlikely
score = dot(customer_tower(customer_features), article_tower(article_features))

# 4. Train: real purchases should have high score, random items low score
model.compile(loss='binary_crossentropy')
model.fit([customer_features, article_features], labels)

# 5. Predict: for each customer, score all 105K articles, take top 12
""")

## 13. Summary — the progression

```
Level 1: Popularity (no ML)
  → Count top 12 most bought articles → recommend to everyone
  → MAP@12 ~0.01-0.02
  → Takes 5 minutes

Level 2: Collaborative Filtering (simple ML)
  → Learn customer + article vectors from purchase history
  → Recommend based on similar customers
  → MAP@12 ~0.02-0.03
  → Takes 30 minutes to train

Level 3: Two-Tower Neural Network (deep learning)
  → Use customer features + article features + purchase history
  → Learn complex patterns
  → MAP@12 ~0.03+
  → Takes hours to train

Level 4: Full system (plan in your Obsidian vault)
  → Multiple candidate generators + ranker + sequence model
  → Best possible score
  → Takes days to build and tune
```

You start at Level 1 and work your way up. Each level adds more complexity but improves the score.